In [1]:
#import data

import pandas as pd

#df = pd.read_parquet("hf://datasets/lmarena-ai/search-arena-24k/data/search-arena-chat-24k.parquet")

from datasets import load_dataset

ds = load_dataset("lmarena-ai/search-arena-24k")

print(ds)


c:\Users\itsav\mambaforge\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    test: Dataset({
        features: ['model_a', 'model_b', 'winner', 'judge', 'turn', 'timestamp', 'messages_a', 'messages_b', 'system_a_metadata', 'system_b_metadata', 'primary_intent', 'secondary_intent', 'languages'],
        num_rows: 24069
    })
})


In [2]:
df = ds["test"].to_pandas()

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
row = df.iloc[0]

print("MODEL A:", row["model_a"])
print("\nMESSAGES A:")
print(row["messages_a"])

print("\n" + "=" * 80)

print("MODEL B:", row["model_b"])
print("\nMESSAGES B:")
print(row["messages_b"])


Shape: (24069, 13)

Columns:
['model_a', 'model_b', 'winner', 'judge', 'turn', 'timestamp', 'messages_a', 'messages_b', 'system_a_metadata', 'system_b_metadata', 'primary_intent', 'secondary_intent', 'languages']
MODEL A: sonar

MESSAGES A:
[{'content': 'what models are you?', 'role': 'user'}
 {'content': 'The term "models" can refer to various types of individuals or entities, depending on the context. Here are a few interpretations:\n\n1. **Fashion Models**: These are individuals who use their physical appearance to promote products or contribute to artistic endeavors. They can be categorized into several types, such as:\n   - **Promotional Models**: Engage in promotional events and interact with consumers to promote products.\n   - **Fashion Models**: Typically work in high-end fashion, gracing runways and posing for editorial shoots in magazines like Vogue.\n   - **Commercial Models**: Appear in advertisements to promote products or services.\n   - **Editorial Models**: Featured in

In [6]:
import re
def count_citations(text):
    if not isinstance(text, str):
        return 0
    
    # Finds citation markers like [1], [2], [12]
    citations = re.findall(r'\[\d+\]', text)
    
    return len(citations)
records = []

for battle_idx, row in df.iterrows():
    
    for side in ["a", "b"]:
        messages = row[f"messages_{side}"]
        model = row[f"model_{side}"]

        # Extract user and assistant messages
        user_messages = [
            m["content"] for m in messages 
            if m["role"] == "user"
        ]

        assistant_messages = [
            m["content"] for m in messages 
            if m["role"] == "assistant"
        ]

        # Skip malformed rows
        if len(assistant_messages) == 0:
            continue

        # Usually one user prompt + one assistant response
        prompt = user_messages[-1] if user_messages else None
        response = assistant_messages[-1]

        records.append({
            "battle_idx": battle_idx,
            "side": side,
            "model": model,
            "prompt": prompt,
            "response": response,
            "winner": row["winner"],
            "primary_intent": row["primary_intent"],
            "secondary_intent": row["secondary_intent"]
        })

responses = pd.DataFrame(records)
responses["citation_count"] = responses["response"].apply(count_citations)

responses["has_citation"] = responses["citation_count"] > 0

In [13]:

zero_citation = responses[
    responses["citation_count"] == 0
]
sample_50 = zero_citation.sample(
    n=50,
    random_state=42
)

#original = df.loc[case["battle_idx"]]

In [14]:
def inspect_case(case_number):
    # case_number is human-readable: 1-50
    case = sample_50.iloc[case_number - 1]

    battle_idx = case["battle_idx"]
    side = case["side"]

    original = df.loc[battle_idx]
    metadata = original[f"system_{side}_metadata"]

    print("=" * 100)
    print(f"CASE {case_number}")
    print("=" * 100)

    print("\nMODEL:")
    print(case["model"])

    print("\nPROMPT:")
    print(case["prompt"])

    print("\nFINAL RESPONSE:")
    print(case["response"])

    print("\n" + "=" * 40)
    print("WEB SEARCH TRACE")
    print("=" * 40)

    print(metadata.get("web_search_trace"))

    print("\n" + "=" * 40)
    print("LLM TRACE")
    print("=" * 40)

    print(metadata.get("llm_trace"))

In [15]:
#Change value to see different zero citation conversations
inspect_case(18)

CASE 18

MODEL:
gpt-4o-search-preview

PROMPT:
碳化矽的應用，碳化矽生產設備製造廠商，台灣的代理商，台灣製造碳化矽的公司

FINAL RESPONSE:
碳化矽（SiC）是一種具有高硬度、高耐熱性和優異電氣特性的材料，廣泛應用於多個領域：

- **半導體產業**：用於製造高功率、高頻率的電子元件，如電動車的功率模組和5G通訊設備。

- **光電產業**：作為LED的基板材料，提升光效和耐用性。

- **機械工業**：製造耐磨損、耐腐蝕的機械零件，如軸承、密封環和噴嘴。

- **化工產業**：用於製造耐高溫、耐腐蝕的化工設備和管道。

在台灣，以下公司專注於碳化矽的生產和相關設備製造：

- **盛新材料科技股份有限公司**：成立於2020年6月，專門生產第三代半導體上游的碳化矽晶錠及基板，已掌握6吋晶錠的生產技術，為台灣首家進入可量產的SiC專業廠。 (taisicmaterials.com)

- **穩晟材料科技股份有限公司**：專業從事碳化矽技術、材料及設備的供應，致力於開發高品質大尺寸碳化矽長晶及晶圓加工技術，推動碳化矽材料的全產業鏈國產化。 (tinva.org)

- **格棋化合物半導體**：成立於2022年，專注於碳化矽長晶生產，旨在補足台灣碳化矽長晶產能的不足，並計劃在未來擴大產能以滿足市場需求。 (mirrormedia.mg)

此外，**申玥科技股份有限公司**代理高品質的碳化矽製品，提供高耐磨性、耐腐蝕的產品，應用於化學槽體、馬達轉子等領域，並協助台灣、韓國與上海的化學廠進行碳化矽部品的更換與維修。 (sy-tech.com.tw)

這些公司在碳化矽材料的生產、加工和應用方面，為台灣的相關產業提供了重要的支持和服務。

WEB SEARCH TRACE
[array([array(['[1]', 'https://www.taisicmaterials.com/'], dtype=object),
        array(['[2]', 'https://tinva.org/calendar/detail/275'], dtype=object),
        array(['[3]', 'https://www.mirrormedia.mg/story/20231127in